# Time Series Patterns

**Core topic**: Feature engineering, forecasting, and stationarity for time series data.

Covers:
1. Sliding window feature extraction (lags, rolling stats)
2. Exponential smoothing (single, double, Holt-Winters)
3. Differencing for stationarity + ADF test concept
4. Autocorrelation (ACF) and partial autocorrelation (PACF)
5. AR model from scratch
6. Forecasting demo on synthetic time series

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## Generate Synthetic Time Series

A series with trend, seasonality, and noise: $y_t = \text{trend}_t + \text{seasonal}_t + \epsilon_t$

In [ ]:
T = 365 * 2  # 2 years of daily data
t = np.arange(T)

# Components
trend = 0.05 * t  # linear trend
seasonal = 10 * np.sin(2 * np.pi * t / 365)  # yearly seasonality
weekly = 3 * np.sin(2 * np.pi * t / 7)  # weekly pattern
noise = np.random.randn(T) * 3

y = trend + seasonal + weekly + noise

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
axes[0].plot(t, y, linewidth=0.5)
axes[0].set_title('Full series (trend + seasonal + weekly + noise)')
axes[1].plot(t, trend)
axes[1].set_title('Trend component')
axes[2].plot(t, seasonal + weekly)
axes[2].set_title('Seasonal components')
axes[3].plot(t, noise, linewidth=0.5)
axes[3].set_title('Noise')
axes[3].set_xlabel('Day')
plt.tight_layout()
plt.show()

## 1. Sliding Window Feature Extraction

The most common way to turn time series into tabular features for ML models.

**Lag features**: $x_{t-1}, x_{t-2}, \ldots, x_{t-k}$

**Rolling statistics**: mean, std, min, max over a rolling window

In [ ]:
def create_lag_features(series, lags):
    """Create lag features. Returns (N - max_lag, len(lags)) array."""
    max_lag = max(lags)
    N = len(series)
    features = np.zeros((N - max_lag, len(lags)))
    for i, lag in enumerate(lags):
        features[:, i] = series[max_lag - lag:N - lag]
    return features


def rolling_stats(series, window):
    """
    Compute rolling mean, std, min, max.
    Returns dict of arrays, each of length len(series) - window + 1.
    """
    N = len(series)
    n_out = N - window + 1
    r_mean = np.zeros(n_out)
    r_std = np.zeros(n_out)
    r_min = np.zeros(n_out)
    r_max = np.zeros(n_out)
    
    for i in range(n_out):
        w = series[i:i + window]
        r_mean[i] = w.mean()
        r_std[i] = w.std()
        r_min[i] = w.min()
        r_max[i] = w.max()
    
    return {'mean': r_mean, 'std': r_std, 'min': r_min, 'max': r_max}


# Demo
lags = [1, 7, 14, 30, 365]
lag_features = create_lag_features(y, lags)
print(f"Lag features shape: {lag_features.shape} (lags: {lags})")

window = 30
r_stats = rolling_stats(y, window)

fig, ax = plt.subplots(figsize=(14, 5))
offset = window - 1
ax.plot(t, y, alpha=0.3, linewidth=0.5, label='raw')
ax.plot(t[offset:], r_stats['mean'], label=f'rolling mean (W={window})', linewidth=2)
ax.fill_between(t[offset:],
                r_stats['mean'] - 2 * r_stats['std'],
                r_stats['mean'] + 2 * r_stats['std'],
                alpha=0.2, label='+/- 2 std')
ax.set_xlabel('Day')
ax.set_title('Rolling Window Features')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Exponential Smoothing

### Single Exponential Smoothing (SES)
For series with no trend or seasonality: $\hat{y}_{t+1} = \alpha y_t + (1-\alpha)\hat{y}_t$

### Double Exponential Smoothing (Holt's method)
Adds a trend component:
- Level: $\ell_t = \alpha y_t + (1-\alpha)(\ell_{t-1} + b_{t-1})$
- Trend: $b_t = \beta (\ell_t - \ell_{t-1}) + (1-\beta) b_{t-1}$
- Forecast: $\hat{y}_{t+h} = \ell_t + h \cdot b_t$

### Triple Exponential Smoothing (Holt-Winters)
Adds seasonality:
- Level: $\ell_t = \alpha (y_t - s_{t-m}) + (1-\alpha)(\ell_{t-1} + b_{t-1})$
- Trend: $b_t = \beta (\ell_t - \ell_{t-1}) + (1-\beta) b_{t-1}$
- Seasonal: $s_t = \gamma (y_t - \ell_t) + (1-\gamma) s_{t-m}$
- Forecast: $\hat{y}_{t+h} = \ell_t + h \cdot b_t + s_{t+h-m}$

where $m$ is the seasonal period.

In [ ]:
def single_exponential_smoothing(series, alpha):
    """SES: for series with no trend/seasonality."""
    result = np.zeros(len(series))
    result[0] = series[0]
    for t in range(1, len(series)):
        result[t] = alpha * series[t] + (1 - alpha) * result[t - 1]
    return result


def double_exponential_smoothing(series, alpha, beta):
    """Holt's method: for series with trend, no seasonality."""
    N = len(series)
    level = np.zeros(N)
    trend_component = np.zeros(N)
    fitted = np.zeros(N)
    
    # Initialize
    level[0] = series[0]
    trend_component[0] = series[1] - series[0] if N > 1 else 0
    fitted[0] = series[0]
    
    for t in range(1, N):
        level[t] = alpha * series[t] + (1 - alpha) * (level[t-1] + trend_component[t-1])
        trend_component[t] = beta * (level[t] - level[t-1]) + (1 - beta) * trend_component[t-1]
        fitted[t] = level[t-1] + trend_component[t-1]  # one-step-ahead forecast
    
    return fitted, level, trend_component


def holt_winters(series, alpha, beta, gamma, season_length, n_forecast=0):
    """
    Triple exponential smoothing (additive Holt-Winters).
    """
    N = len(series)
    m = season_length
    
    # Initialize level and trend from first season
    level = np.zeros(N)
    trend_comp = np.zeros(N)
    seasonal = np.zeros(N + n_forecast)
    fitted = np.zeros(N)
    
    # Initial values
    level[0] = np.mean(series[:m])
    trend_comp[0] = (np.mean(series[m:2*m]) - np.mean(series[:m])) / m if N >= 2*m else 0
    
    # Initialize seasonal components from first season
    for i in range(m):
        seasonal[i] = series[i] - level[0]
    
    fitted[0] = level[0] + seasonal[0]
    
    for t in range(1, N):
        if t >= m:
            level[t] = alpha * (series[t] - seasonal[t - m]) + (1 - alpha) * (level[t-1] + trend_comp[t-1])
        else:
            level[t] = alpha * (series[t] - seasonal[t % m]) + (1 - alpha) * (level[t-1] + trend_comp[t-1])
        
        trend_comp[t] = beta * (level[t] - level[t-1]) + (1 - beta) * trend_comp[t-1]
        
        if t >= m:
            seasonal[t] = gamma * (series[t] - level[t]) + (1 - gamma) * seasonal[t - m]
        
        # One-step-ahead forecast
        s_idx = t - m if t >= m else t % m
        fitted[t] = level[t-1] + trend_comp[t-1] + seasonal[s_idx]
    
    # Generate forecasts
    forecasts = np.zeros(n_forecast)
    for h in range(1, n_forecast + 1):
        s_idx = N - m + ((h - 1) % m)
        forecasts[h - 1] = level[N - 1] + h * trend_comp[N - 1] + seasonal[s_idx]
    
    return fitted, forecasts, level, trend_comp, seasonal[:N]


# Demo on our synthetic series (use shorter version for clarity)
y_short = y[:365]  # one year

ses_fit = single_exponential_smoothing(y_short, alpha=0.1)
des_fit, des_level, des_trend = double_exponential_smoothing(y_short, alpha=0.3, beta=0.1)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(y_short, alpha=0.4, linewidth=0.5, label='actual')
axes[0].plot(ses_fit, label='SES (alpha=0.1)', linewidth=2)
axes[0].set_title('Single Exponential Smoothing')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(y_short, alpha=0.4, linewidth=0.5, label='actual')
axes[1].plot(des_fit, label='Double ES (alpha=0.3, beta=0.1)', linewidth=2)
axes[1].set_title('Double Exponential Smoothing (Holt\'s Method)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Holt-Winters with yearly seasonality
train = y[:365]
test = y[365:365+90]  # forecast 90 days

hw_fit, hw_forecast, hw_level, hw_trend, hw_seasonal = holt_winters(
    train, alpha=0.2, beta=0.05, gamma=0.1, season_length=7, n_forecast=90
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(range(len(train)), train, alpha=0.4, linewidth=0.5, label='train')
ax.plot(range(len(train)), hw_fit, label='Holt-Winters fit', linewidth=1.5)
ax.plot(range(len(train), len(train) + 90), test, alpha=0.6, label='actual (test)', linewidth=1.5)
ax.plot(range(len(train), len(train) + 90), hw_forecast, '--', label='forecast', linewidth=2)
ax.axvline(len(train), color='black', linestyle=':', alpha=0.5)
ax.set_xlabel('Day')
ax.set_title('Holt-Winters Forecasting')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Differencing for Stationarity

**Stationarity**: A series is stationary if its statistical properties (mean, variance, autocorrelation) do not change over time.

**Why it matters**: Most classical models (ARMA/ARIMA, etc.) assume stationarity. Non-stationary data leads to spurious correlations and unreliable forecasts.

**Differencing**: $\Delta y_t = y_t - y_{t-1}$ removes linear trend. Apply twice for quadratic trend. Seasonal differencing: $\Delta_m y_t = y_t - y_{t-m}$.

**ADF Test concept**: Augmented Dickey-Fuller test.
- $H_0$: Series has a unit root (non-stationary)
- $H_1$: Series is stationary
- Fits regression: $\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum \delta_i \Delta y_{t-i} + \epsilon_t$
- Test statistic: t-ratio on $\gamma$. If sufficiently negative, reject $H_0$.

In [ ]:
def difference(series, d=1):
    """Apply d-th order differencing."""
    result = series.copy()
    for _ in range(d):
        result = np.diff(result)
    return result


def seasonal_difference(series, period):
    """Seasonal differencing: y_t - y_{t-period}."""
    return series[period:] - series[:-period]


def simple_adf_statistic(series, max_lag=5):
    """
    Simplified ADF test statistic (without proper critical values).
    Tests H0: unit root (non-stationary) vs H1: stationary.
    More negative = more evidence for stationarity.
    Rough critical values at 5%: -2.86 (no trend), -3.41 (with trend).
    """
    dy = np.diff(series)
    N = len(dy)
    
    # Build regression: dy_t = alpha + gamma * y_{t-1} + sum(delta_i * dy_{t-i}) + eps
    # We want the t-stat on gamma
    y_lag = series[max_lag:-1]  # y_{t-1}
    dy_target = dy[max_lag:]    # dy_t
    
    # Build design matrix: [1, y_{t-1}, dy_{t-1}, ..., dy_{t-max_lag}]
    n = len(dy_target)
    X = np.ones((n, 2 + max_lag))
    X[:, 1] = y_lag
    for lag in range(1, max_lag + 1):
        X[:, 1 + lag] = dy[max_lag - lag:N - lag]
    
    # OLS
    beta_hat = np.linalg.lstsq(X, dy_target, rcond=None)[0]
    residuals = dy_target - X @ beta_hat
    sigma2 = np.sum(residuals**2) / (n - X.shape[1])
    var_beta = sigma2 * np.linalg.inv(X.T @ X)
    
    # t-statistic on gamma (coefficient of y_{t-1})
    gamma_hat = beta_hat[1]
    se_gamma = np.sqrt(var_beta[1, 1])
    t_stat = gamma_hat / se_gamma
    
    return t_stat


# Demo
y_diff1 = difference(y, d=1)
y_diff_seasonal = seasonal_difference(y, period=7)

fig, axes = plt.subplots(3, 1, figsize=(14, 9))

axes[0].plot(y, linewidth=0.5)
adf_orig = simple_adf_statistic(y)
axes[0].set_title(f'Original series (ADF stat = {adf_orig:.2f})')

axes[1].plot(y_diff1, linewidth=0.5)
adf_diff = simple_adf_statistic(y_diff1)
axes[1].set_title(f'First difference (ADF stat = {adf_diff:.2f})')

axes[2].plot(y_diff_seasonal, linewidth=0.5)
adf_seas = simple_adf_statistic(y_diff_seasonal)
axes[2].set_title(f'Seasonal difference (period=7, ADF stat = {adf_seas:.2f})')

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("ADF interpretation: more negative = more evidence for stationarity")
print("Rough 5% critical value: -2.86 (reject H0 if stat < -2.86)")

## 4. Autocorrelation (ACF) and Partial Autocorrelation (PACF)

**ACF at lag k**: Correlation between $y_t$ and $y_{t-k}$
$$\rho(k) = \frac{\text{Cov}(y_t, y_{t-k})}{\text{Var}(y_t)}$$

**PACF at lag k**: Correlation between $y_t$ and $y_{t-k}$ after removing the effect of intermediate lags. Computed via sequential regression (Durbin-Levinson).

**Interpretation for model selection**:
- ACF cuts off at lag q, PACF decays -> MA(q) model
- ACF decays, PACF cuts off at lag p -> AR(p) model
- Both decay -> ARMA(p,q) model

In [ ]:
def acf(series, max_lag=40):
    """Compute autocorrelation function."""
    N = len(series)
    mean = np.mean(series)
    var = np.var(series)
    acf_values = np.zeros(max_lag + 1)
    
    for k in range(max_lag + 1):
        if k == 0:
            acf_values[k] = 1.0
        else:
            cov = np.mean((series[k:] - mean) * (series[:-k] - mean))
            acf_values[k] = cov / var
    
    return acf_values


def pacf(series, max_lag=40):
    """
    Compute partial autocorrelation using OLS regression.
    PACF at lag k = coefficient of y_{t-k} in regression of y_t on y_{t-1},...,y_{t-k}.
    """
    N = len(series)
    pacf_values = np.zeros(max_lag + 1)
    pacf_values[0] = 1.0
    
    for k in range(1, max_lag + 1):
        # Build design matrix with lags 1..k
        y_target = series[k:]
        n = len(y_target)
        X = np.ones((n, k + 1))  # intercept + k lags
        for lag in range(1, k + 1):
            X[:, lag] = series[k - lag:N - lag]
        
        beta = np.linalg.lstsq(X, y_target, rcond=None)[0]
        pacf_values[k] = beta[k]  # coefficient of the k-th lag
    
    return pacf_values


# Compute on differenced series (stationary)
y_stat = difference(y, d=1)
acf_vals = acf(y_stat, max_lag=50)
pacf_vals = pacf(y_stat, max_lag=30)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lags_acf = np.arange(len(acf_vals))
axes[0].bar(lags_acf, acf_vals, width=0.3, color='steelblue')
conf = 1.96 / np.sqrt(len(y_stat))
axes[0].axhline(conf, color='red', linestyle='--', alpha=0.5)
axes[0].axhline(-conf, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('ACF')
axes[0].set_title('Autocorrelation Function (differenced series)')
axes[0].grid(True, alpha=0.3)

lags_pacf = np.arange(len(pacf_vals))
axes[1].bar(lags_pacf, pacf_vals, width=0.3, color='darkorange')
axes[1].axhline(conf, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(-conf, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('PACF')
axes[1].set_title('Partial Autocorrelation Function (differenced series)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: significant PACF spikes at lags 1, 7 suggest AR terms with those lags.")
print("Significant ACF at lag 7 confirms weekly seasonality.")

## 5. AR Model from Scratch

**AR(p)**: $y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \epsilon_t$

Fit by OLS: regress $y_t$ on its p lagged values.

In [ ]:
class ARModel:
    """Autoregressive model of order p, fit by OLS."""
    
    def __init__(self, p):
        self.p = p
        self.coefficients = None  # [intercept, phi_1, ..., phi_p]
    
    def fit(self, series):
        """Fit AR(p) model by OLS."""
        N = len(series)
        y = series[self.p:]  # target
        n = len(y)
        
        # Design matrix: [1, y_{t-1}, y_{t-2}, ..., y_{t-p}]
        X = np.ones((n, self.p + 1))
        for lag in range(1, self.p + 1):
            X[:, lag] = series[self.p - lag:N - lag]
        
        # OLS: beta = (X^T X)^{-1} X^T y
        self.coefficients = np.linalg.lstsq(X, y, rcond=None)[0]
        
        # Fitted values and residuals
        fitted = X @ self.coefficients
        residuals = y - fitted
        self.sigma2 = np.var(residuals)
        
        return fitted, residuals
    
    def predict(self, series, n_forecast):
        """Recursive multi-step forecast."""
        # Start from the last p values
        history = list(series[-self.p:])
        forecasts = []
        
        for _ in range(n_forecast):
            x = np.array([1.0] + history[-self.p:][::-1])  # [1, y_{t-1}, y_{t-2}, ...]
            # Fix: we need [1, y_{t-1}, ..., y_{t-p}]
            x = np.array([1.0] + list(reversed(history[-self.p:])))
            y_hat = x @ self.coefficients
            forecasts.append(y_hat)
            history.append(y_hat)
        
        return np.array(forecasts)


# Fit AR on differenced series
y_stat = difference(y, d=1)
train_stat = y_stat[:365]
test_stat = y_stat[365:365+60]

ar = ARModel(p=7)  # AR(7) to capture weekly pattern
fitted, residuals = ar.fit(train_stat)

print("AR(7) coefficients:")
print(f"  intercept: {ar.coefficients[0]:.4f}")
for i in range(1, ar.p + 1):
    print(f"  phi_{i}: {ar.coefficients[i]:.4f}")
print(f"  residual variance: {ar.sigma2:.4f}")

In [ ]:
# Forecast and convert back from differences to levels
forecast_diff = ar.predict(train_stat, n_forecast=60)

# Cumulative sum to convert back: y_t = y_0 + sum(diff_1..t)
last_value = y[365]  # last actual value before forecast
forecast_levels = last_value + np.cumsum(forecast_diff)
actual_test = y[366:366+60]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Differenced series fit
axes[0].plot(train_stat, alpha=0.4, linewidth=0.5, label='actual (diff)')
axes[0].plot(range(ar.p, ar.p + len(fitted)), fitted, label='AR(7) fit', linewidth=1.5)
axes[0].set_title('AR(7) Fit on Differenced Series')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Level forecast
n_show = 120
axes[1].plot(range(365-n_show, 366), y[365-n_show:366], alpha=0.6, label='actual (train)', linewidth=1)
axes[1].plot(range(366, 366+60), actual_test, label='actual (test)', linewidth=1.5)
axes[1].plot(range(366, 366+60), forecast_levels, '--', label='AR(7) forecast', linewidth=2)
axes[1].axvline(365, color='black', linestyle=':', alpha=0.5)
axes[1].set_title('AR(7) Forecast (converted back to levels)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('Day')

plt.tight_layout()
plt.show()

## 6. Time Series Decomposition

Additive decomposition: $y_t = T_t + S_t + R_t$ (Trend + Seasonal + Residual)

Simple approach: moving average for trend, average by position for seasonal.

In [ ]:
def decompose(series, period):
    """Simple additive decomposition."""
    N = len(series)
    
    # Trend: centered moving average
    if period % 2 == 0:
        # For even period, use 2x moving average
        kernel = np.ones(period) / period
        trend_raw = np.convolve(series, kernel, mode='valid')
        # Center it
        offset = period // 2
        trend = np.full(N, np.nan)
        trend[offset:offset + len(trend_raw)] = trend_raw
    else:
        kernel = np.ones(period) / period
        trend_raw = np.convolve(series, kernel, mode='valid')
        offset = period // 2
        trend = np.full(N, np.nan)
        trend[offset:offset + len(trend_raw)] = trend_raw
    
    # Detrended
    detrended = series - trend
    
    # Seasonal: average detrended values by position in cycle
    seasonal = np.zeros(N)
    for i in range(period):
        mask = np.arange(i, N, period)
        vals = detrended[mask]
        valid = vals[~np.isnan(vals)]
        if len(valid) > 0:
            seasonal[mask] = np.nanmean(valid)
    
    # Residual
    residual = series - trend - seasonal
    
    return trend, seasonal, residual


trend_est, seasonal_est, residual_est = decompose(y[:365], period=7)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

axes[0].plot(y[:365], linewidth=0.5)
axes[0].set_title('Observed')

axes[1].plot(trend_est, linewidth=2)
axes[1].set_title('Trend (7-day moving average)')

axes[2].plot(seasonal_est, linewidth=1)
axes[2].set_title('Seasonal (weekly pattern)')

axes[3].plot(residual_est, linewidth=0.5)
axes[3].set_title('Residual')
axes[3].set_xlabel('Day')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interview Questions & Answers

### "How to create features from time series?"
1. **Lag features**: Past values as predictors (y_{t-1}, y_{t-7}, etc.)
2. **Rolling statistics**: Mean, std, min, max over a sliding window
3. **Time-based features**: Hour of day, day of week, month, holiday flags
4. **Differencing features**: First difference, seasonal difference
5. **Fourier features**: Sine/cosine terms at known frequencies for capturing seasonality

Key: Avoid data leakage -- features must use only past data at each prediction time.

### "Stationarity -- why does it matter?"
- Stationarity means statistical properties are constant over time.
- Non-stationary series have time-varying mean/variance -> correlations are spurious.
- ARMA models assume stationarity. Violating this leads to unreliable parameter estimates.
- Fix: difference the series (ARIMA's "I" stands for "integrated" = differenced).
- Test: ADF test. Null = unit root (non-stationary). Reject if test stat is sufficiently negative.

### "Walk me through Holt-Winters"
Three components, each with its own smoothing parameter:
1. **Level** (alpha): Smoothed value after removing seasonality
2. **Trend** (beta): Smoothed rate of change in the level
3. **Seasonal** (gamma): Repeating pattern at period m

Forecast h steps ahead: level + h * trend + seasonal_{t+h mod m}. The additive version assumes constant seasonal amplitude; multiplicative version handles growing amplitude.

## Modern Approaches (Notes)

| Method | Key Idea | When to Use |
|--------|----------|-------------|
| **Prophet** (Meta) | Additive model with trend + seasonality + holidays | Business time series, interpretability needed |
| **N-BEATS** | Pure DL, basis expansion using stacks of FC layers | Univariate forecasting, no exogenous features |
| **Temporal Fusion Transformer** | Attention over time + variable selection | Multi-horizon with static/dynamic covariates |
| **PatchTST** | Transformer on patched time series segments | Long-horizon, channel-independent |
| **TimesFM** (Google) | Foundation model pre-trained on time series | Zero-shot forecasting |